# RDD Baseline — Naive Bayes Classifier

**Unoptimised MapReduce implementation using the PySpark RDD API.**

This notebook is the baseline reference — no optimisations applied.
It uses `flatMap` (one emit per feature per row) and `groupByKey`
(full shuffle before any reduction). These choices are intentionally
inefficient to establish a lower-bound on performance.
See `naive_bayes_rdd_optimized.ipynb` for the improved version.

Reference: Zheng, S. (2014). *Naïve Bayes Classifier: A MapReduce Approach*. NDSU.

In [ ]:
# Cell 1 — Imports and SparkSession setup
import time
import math
import sys
import os

from pyspark.sql import SparkSession

# Add the project root to sys.path so we can import from core/ and data/.
# This works when the notebook is in rdd/; adjust if you move it.
sys.path.insert(0, os.path.abspath('..'))

from core.naive_bayes import compute_log_probs, predict, evaluate
from data.loader import load_rdd, FEATURE_COLS, LABEL_COL

# TODO (Databricks): Remove the .master() line below — the cluster provides
# its own master URL. On Databricks, SparkSession is already available as
# the global `spark` variable and getOrCreate() will return it.
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('NaiveBayes-RDD-Baseline')
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('WARN')  # suppress INFO noise during development
print('SparkSession ready.')

In [ ]:
# Cell 2 — Load data
#
# TODO: Replace None with the path to your dataset file.
#   Local       : '/Users/you/data/car.data'
#   Databricks  : 'dbfs:/FileStore/car.data'
# Leave as None to run immediately with the 5-row dummy dataset.
DATA_PATH = None

train_rdd, test_rdd = load_rdd(spark, filepath=DATA_PATH)

# Print basic stats so we can verify the data loaded correctly before training.
train_count = train_rdd.count()
test_count  = test_rdd.count()
print(f'Train rows : {train_count}')
print(f'Test rows  : {test_count}')
print(f'Sample row : {train_rdd.first()}')

# Show class distribution in training data to confirm the dataset loaded correctly.
# We expect ~70% unacc, ~22% acc, ~4% good, ~4% vgood.
class_dist = train_rdd.map(lambda row: row[0]).countByValue()
print(f'Class distribution: {dict(class_dist)}')

In [ ]:
# Cell 3 — Training phase (MapReduce)
#
# BASELINE BEHAVIOUR — deliberately unoptimised:
#   - flatMap emits one (key, 1) pair per feature per row (no local aggregation)
#   - groupByKey shuffles ALL (key, 1) pairs to a single node before summing
#   - No broadcast, no persist, no repartition

t_train_start = time.time()

# MAP PHASE
# Each training row emits:
#   - One pair per feature: ("feat_{i}_{value}_{class}", 1)
#   - One class pair: ("class_{label}", 1)
#   - One total pair: ("total", 1)
# Emitting 1 for every single occurrence (not aggregating locally) is what
# makes this the baseline — all aggregation happens in the shuffle.
#
# Input row:  ('unacc', ['low', 'med', '2', '2', 'small', 'low'])
# Emitted  :  [('feat_0_low_unacc', 1), ('feat_1_med_unacc', 1), ...,
#              ('class_unacc', 1), ('total', 1)]
def map_row(row):
    label, features = row
    pairs = []
    for feat_idx, value in enumerate(features):
        pairs.append((f'feat_{feat_idx}_{value}_{label}', 1))
    pairs.append((f'class_{label}', 1))
    pairs.append(('total', 1))
    return pairs

# flatMap applies map_row to every row and flattens the list-of-lists.
# Input:  RDD of (label, [features])
# Output: RDD of ('feat_i_value_class', 1) and ('class_label', 1) and ('total', 1)
mapped_rdd = train_rdd.flatMap(map_row)

# REDUCE PHASE — groupByKey (BASELINE: intentionally inefficient)
# groupByKey collects ALL values for each key into an iterable BEFORE summing.
# This means every single (key, 1) pair is shuffled across the network first,
# then summed on one node. No partial reduction happens on the mapper side.
# Input:  ('feat_0_low_unacc', 1), ('feat_0_low_unacc', 1), ...
# Output: ('feat_0_low_unacc', <iterable of 1s>)
grouped_rdd = mapped_rdd.groupByKey()

# Sum the iterable of 1s to get the final count for each key.
# Input:  ('feat_0_low_unacc', [1, 1, 1])
# Output: ('feat_0_low_unacc', 3)
counts_rdd = grouped_rdd.mapValues(sum)

# Collect all counts to the driver.
# The result is O(classes * features * unique_values) — small enough for driver memory.
all_counts = dict(counts_rdd.collect())

train_time = time.time() - t_train_start
print(f'[TIMING] Training: {train_time:.4f}s')
print(f'Total unique keys : {len(all_counts)}')
print(f'Sample counts     : {list(all_counts.items())[:4]}')

In [ ]:
# Cell 4 — Probability computation
#
# We separate the raw counts from the probability computation so that
# compute_log_probs() can be shared across both RDD and DataFrame implementations.
# This is the only cell that imports from core/naive_bayes.py.

# Separate the mixed all_counts dict into the three input types
# that compute_log_probs() expects.
class_counts   = {k[6:]: v for k, v in all_counts.items() if k.startswith('class_')}
feature_counts = {k: v       for k, v in all_counts.items() if k.startswith('feat_')}
class_totals   = class_counts  # for standard Naive Bayes these are the same

log_prob_table = compute_log_probs(class_counts, feature_counts, class_totals)

# Spot-check the probability table to verify it looks reasonable.
# Log-probabilities should be negative (since all P < 1).
print(f"Classes           : {log_prob_table['classes']}")
print(f"Log class probs   : {log_prob_table['log_class_probs']}")
print(f"Sample feat probs : {list(log_prob_table['log_feature_probs'].items())[:3]}")

In [ ]:
# Cell 5 — Prediction
#
# BASELINE BEHAVIOUR:
#   log_prob_table is NOT broadcast — Spark serialises it with every task.
#   On a large cluster with many tasks this is wasteful; see the optimised
#   notebook for the broadcast version.

t_pred_start = time.time()

# Apply predict() from core/naive_bayes.py to every test row.
# The closure captures log_prob_table from the driver scope — Spark will
# serialise and ship it with each task individually (no broadcast).
# Input row:  ('unacc', ['low', 'med', '2', '2', 'small', 'low'])
# Output   :  ('unacc', 'unacc')  <- (true_label, predicted_label)
predictions_rdd = test_rdd.map(
    lambda row: (row[0], predict(log_prob_table, row[1]))
)

# Collect forces evaluation (Spark is lazy — nothing runs until an action is called).
predictions_list = predictions_rdd.collect()

pred_time = time.time() - t_pred_start
print(f'[TIMING] Prediction: {pred_time:.4f}s')
print(f'Sample predictions : {predictions_list[:5]}')

In [ ]:
# Cell 6 — Evaluation

true_labels  = [pair[0] for pair in predictions_list]
pred_labels  = [pair[1] for pair in predictions_list]

accuracy = evaluate(pred_labels, true_labels)

print(f'Accuracy     : {accuracy:.4f} ({accuracy * 100:.2f}%)')
print(f'Train time   : {train_time:.4f}s')
print(f'Predict time : {pred_time:.4f}s')
print(f'Total time   : {train_time + pred_time:.4f}s')

# TODO: Expected accuracy on the full car evaluation dataset is ~87.39%
# (Zheng 2014). With the 5-row dummy dataset accuracy is not meaningful —
# replace DATA_PATH with the real file path before interpreting results.